# Cleaning 2.2 - Clean unique values in panel dataset

In [1]:
# Indicator for new variables to be cleaned (default is False, set to True if decide to clean more variables)
new_vars_to_clean = False

In [2]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from create_empty_cleaning_sheet import create_empty_cleaning_sheet
from unique_values_cleaning import clean_unique_values

In [3]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_1.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [4]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

In [5]:
# Create or update the excel workbook with empty sheets
if new_vars_to_clean:

    # File name
    file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

    # Equipment questions (one sheet per variable)
    for _, row in equipment_data_dict.iterrows():   
        sheet_name = row["Variable"]
        free_text = row["Free text"] == "Y"
        mc_fc_vars = free_text
        create_empty_cleaning_sheet(file_name, sheet_name, 
                                    comment = True, 
                                    free_text = free_text, 
                                    mc_fc_vars = mc_fc_vars)

In [6]:
# Run cleaning loop. This will: 
# 1. Merge our data to the cleaning workbook
# 2. Create cleaned variables
# 3. produce a report of the cleaning process
# 4. Update the cleaning workbook with all uncleaned values

file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

# Equipment questions (one sheet per variable)
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    sheet_name = var_name
    free_text = row["Free text"] == "Y"
    mc_fc_vars = free_text
    equipment=clean_unique_values(df=equipment, file_name=file_name, 
                                  sheet_name=sheet_name, var_name=var_name, 
                                  comment=True, free_text=free_text, 
                                  mc_fc_vars=mc_fc_vars, 
                                  report=True)

Cleaning progress for controller_type:
Total unique value combinations: 28
Cleaned combinations: 25
Pending combinations: 0
Excluded combinations: 3
Unchecked combinations: 0
Cleaning progress for sash_width:
Total unique value combinations: 44
Cleaned combinations: 42
Pending combinations: 0
Excluded combinations: 1
Unchecked combinations: 1
Cleaning progress for face_velocity:
Total unique value combinations: 60
Cleaned combinations: 57
Pending combinations: 0
Excluded combinations: 2
Unchecked combinations: 1
Cleaning progress for number:
Total unique value combinations: 94
Cleaned combinations: 88
Pending combinations: 2
Excluded combinations: 3
Unchecked combinations: 1
Cleaning progress for lifted:
Total unique value combinations: 13
Cleaned combinations: 13
Pending combinations: 0
Excluded combinations: 0
Unchecked combinations: 0
Cleaning progress for hours_open:
Total unique value combinations: 29
Cleaned combinations: 28
Pending combinations: 0
Excluded combinations: 1
Unchec

In [7]:
# For "pending" values, print the labgroupids, survey and variable names and value so that we can check the surveys

# Equipment questions - print labgroupids and survey for pending values
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    for s in ["BL", "EL"]:
        for e in ["fc", "fridge", "freezer", "ult", "glassware", "microbio", "cryostat", "bath", "incubator", "heater", "it"]:
            pending_values = equipment[
                (equipment[f"{var_name}_status"] == "Pending") &
                (equipment["survey"] == s) & 
                (equipment["equipment"] == e)
            ]["labgroupid"].tolist()
            if pending_values:
                print(f"Pending values for {var_name}, survey {s}, equipment {e}: {pending_values}")


Pending values for number, survey EL, equipment it: [646, 646]
Pending values for capacity_glassware, survey BL, equipment glassware: [759, 700, 559, 722, 177, 414, 414, 214]
Pending values for capacity_glassware, survey EL, equipment glassware: [759, 700, 559, 722, 177, 414, 414, 214]
Pending values for tech, survey BL, equipment glassware: [414]
Pending values for tech, survey EL, equipment glassware: [414, 414]
Pending values for fan, survey BL, equipment glassware: [330]
Pending values for fan, survey EL, equipment glassware: [330]
Pending values for temp_glassware, survey BL, equipment glassware: [388, 177, 588, 646, 820, 956]
Pending values for temp_glassware, survey EL, equipment glassware: [388, 177, 588, 646, 820, 956]
Pending values for hours, survey BL, equipment glassware: [910, 956]
Pending values for hours, survey BL, equipment microbio: [705]
Pending values for hours, survey BL, equipment heater: [700, 281, 642, 956]
Pending values for hours, survey BL, equipment it: [13

In [8]:
# For FCs, view all the relevant columns to identify gaps in the data

# List of FC variables (controller_type, sash_width, face_velocity, number, lifted, hours_open, days, surface)
fc_vars = ["controller_type", "sash_width", "face_velocity", "number", "lifted", "hours_open", "days", "surface"]

fc_data = equipment[equipment["equipment"] == "fc"]

# Keep only relevant cols for FC variables
cols_to_keep = ["labgroupid", "survey"] + [f"{var}_clean" for var in fc_vars]
fc_data = fc_data[cols_to_keep]

In [9]:
# For ULTs, view all the relevant columns to identify gaps in the data
ult_data = equipment[equipment["equipment"] == "ult"]
# Keep only relevant cols for ULT variables (ult_type, size_ult, temp_ult, number, seals, spacing, filter, door_openings)
ult_vars = ["ult_type", "size_ult", "temp_ult", "number", "seals", "spacing", "filter", "door_openings"]
cols_to_keep_ult = ["labgroupid", "survey"] + [f"{var}_clean" for var in ult_vars]
ult_data = ult_data[cols_to_keep_ult]